# 02_recolectar_respuestas_x - plan formal seguro

## Objetivo
Preparar la recoleccion de replies y quote tweets asociados a los posts madre formales de medios costarricenses.

Por defecto este notebook funciona en modo **dry-run**: deduplica posts, conserva las membresias de eventos, construye queries, estima solicitudes y guarda manifiestos. No usa el Bearer Token ni llama a la API de X mientras `EXECUTE_X_API_COLLECTION=false`.


## Entradas
- `data/interim/source_posts_candidates_formal.csv`
- `config/events.yaml`
- `.env` solo cuando se active una recoleccion futura

## Salidas del dry-run
- `data/interim/source_posts_formal_unique.csv`
- `data/interim/reply_collection_manifest_formal.csv`
- `data/interim/quote_collection_manifest_formal.csv`
- `outputs/tables/interaction_collection_plan_formal.csv`
- `outputs/tables/interaction_collection_batches_formal.csv`

## Salidas futuras de API
Solo si se supera el bloqueo de seguridad, los CSV se escribiran en `data/interim/formal_collection/{batch_id}/` y los JSONL crudos en `data/raw/formal_collection/{batch_id}/`.


## 1. Setup


In [ ]:
import os
import sys
import importlib
from pathlib import Path

import pandas as pd
import yaml
from dotenv import load_dotenv
from IPython.display import display


def is_project_root(path):
    return (path / "config").exists() and (path / "src").exists() and (path / "notebooks").exists()


def find_project_root(start):
    for candidate in [start] + list(start.parents):
        if is_project_root(candidate):
            return candidate
        child = candidate / "HateCR"
        if is_project_root(child):
            return child
    raise FileNotFoundError("No se encontro la raiz del proyecto HateCR")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env", override=False)

import src.x_api as xapi
import src.collection as col

importlib.reload(xapi)
importlib.reload(col)

CONFIG_DIR = PROJECT_ROOT / "config"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
FORMAL_COLLECTION_DIR = INTERIM_DIR / "formal_collection"
FORMAL_RAW_DIR = RAW_DIR / "formal_collection"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)
FORMAL_COLLECTION_DIR.mkdir(parents=True, exist_ok=True)
FORMAL_RAW_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)


## 2. Parametros y bloqueo de seguridad


In [ ]:
RUN_MODE = os.getenv("RUN_MODE", "formal_dry_run").strip().lower()
COLLECTION_SCOPE = os.getenv("COLLECTION_SCOPE", "media_anchored").strip().lower()
USE_FULL_ARCHIVE = os.getenv("USE_FULL_ARCHIVE", "true").strip().lower() in {"1", "true", "yes", "si"}
ALLOW_RECENT_FALLBACK = False

EXECUTE_X_API_COLLECTION = os.getenv("EXECUTE_X_API_COLLECTION", "false").strip().lower() == "true"
EXECUTION_CONFIRMATION = os.getenv("CONFIRM_FORMAL_COLLECTION", "").strip()
REQUIRED_CONFIRMATION = "HATECR_EXECUTE_FORMAL_X_API"
COLLECTION_BATCH_ID = os.getenv("COLLECTION_BATCH_ID", "").strip()
ALLOW_ALL_BATCHES = os.getenv("ALLOW_ALL_BATCHES", "false").strip().lower() == "true"
ALLOW_BATCH_OVERWRITE = os.getenv("ALLOW_BATCH_OVERWRITE", "false").strip().lower() == "true"

ENABLE_REPLIES_TO_MEDIA_POSTS = os.getenv("ENABLE_REPLIES_TO_MEDIA_POSTS", "true").strip().lower() == "true"
ENABLE_QUOTES_OF_MEDIA_POSTS = os.getenv("ENABLE_QUOTES_OF_MEDIA_POSTS", "false").strip().lower() == "true"

EVENT_ID_FILTER = [
    value.strip()
    for value in os.getenv("EVENT_ID_FILTER", "").split(",")
    if value.strip()
]
MEDIA_ID_FILTER = [
    value.strip()
    for value in os.getenv("MEDIA_ID_FILTER", "").split(",")
    if value.strip()
]

MIN_SOURCE_REPLY_COUNT = int(os.getenv("MIN_SOURCE_REPLY_COUNT", "1"))
MIN_SOURCE_QUOTE_COUNT = int(os.getenv("MIN_SOURCE_QUOTE_COUNT", "1"))
MAX_POSTS_PER_EVENT_MEDIA = int(os.getenv("MAX_POSTS_PER_EVENT_MEDIA", "10"))
COLLECTION_BATCH_SIZE = int(os.getenv("COLLECTION_BATCH_SIZE", "50"))

REPLY_WINDOW_HOURS = int(os.getenv("REPLY_WINDOW_HOURS", "72"))
MAX_REPLIES_PER_POST = int(os.getenv("MAX_REPLIES_PER_POST", "100"))
MAX_PAGES_PER_POST = int(os.getenv("MAX_PAGES_PER_POST", "10"))
MAX_RESULTS_PER_REPLY_PAGE = int(os.getenv("MAX_RESULTS_PER_REPLY_PAGE", "100"))

MAX_QUOTES_PER_POST = int(os.getenv("MAX_QUOTES_PER_POST", "100"))
MAX_QUOTE_PAGES = int(os.getenv("MAX_QUOTE_PAGES", "10"))
MAX_RESULTS_PER_QUOTE_PAGE = int(os.getenv("MAX_RESULTS_PER_QUOTE_PAGE", "100"))

CHECKPOINT_EVERY = int(os.getenv("CHECKPOINT_EVERY", "10"))
X_TIMEOUT_SECONDS = int(os.getenv("X_TIMEOUT_SECONDS", "60"))
X_SLEEP_SECONDS = float(os.getenv("X_SLEEP_SECONDS", "0.25"))
X_MAX_RATE_WAIT_SECONDS = int(os.getenv("X_MAX_RATE_WAIT_SECONDS", "60"))
X_MAX_429_RETRIES = int(os.getenv("X_MAX_429_RETRIES", "3"))
STOP_ON_API_ERROR = False

assert COLLECTION_SCOPE == "media_anchored", "Este notebook formal exige COLLECTION_SCOPE=media_anchored"
assert USE_FULL_ARCHIVE, "La recoleccion formal historica exige USE_FULL_ARCHIVE=true"

if EXECUTE_X_API_COLLECTION and EXECUTION_CONFIRMATION != REQUIRED_CONFIRMATION:
    raise RuntimeError(
        "Ejecucion API bloqueada: falta CONFIRM_FORMAL_COLLECTION=" + REQUIRED_CONFIRMATION
    )
if EXECUTE_X_API_COLLECTION and not COLLECTION_BATCH_ID and not ALLOW_ALL_BATCHES:
    raise RuntimeError(
        "Ejecucion API bloqueada: indica COLLECTION_BATCH_ID o activa ALLOW_ALL_BATCHES=true"
    )

API_EXECUTION_ALLOWED = (
    EXECUTE_X_API_COLLECTION
    and EXECUTION_CONFIRMATION == REQUIRED_CONFIRMATION
    and (bool(COLLECTION_BATCH_ID) or ALLOW_ALL_BATCHES)
)

print("RUN_MODE:", RUN_MODE)
print("COLLECTION_SCOPE:", COLLECTION_SCOPE)
print("EXECUTE_X_API_COLLECTION:", EXECUTE_X_API_COLLECTION)
print("API_EXECUTION_ALLOWED:", API_EXECUTION_ALLOWED)
print("COLLECTION_BATCH_ID:", COLLECTION_BATCH_ID or "NO DEFINIDO")
print("ENABLE_REPLIES_TO_MEDIA_POSTS:", ENABLE_REPLIES_TO_MEDIA_POSTS)
print("ENABLE_QUOTES_OF_MEDIA_POSTS:", ENABLE_QUOTES_OF_MEDIA_POSTS)
print("MAX_POSTS_PER_EVENT_MEDIA:", MAX_POSTS_PER_EVENT_MEDIA)
print("REPLY_WINDOW_HOURS:", REPLY_WINDOW_HOURS)


## 3. Carga de los posts madre formales


In [ ]:
with open(CONFIG_DIR / "events.yaml", "r", encoding="utf-8") as file:
    events_cfg = yaml.safe_load(file) or {}

formal_events = sorted(
    [
        event
        for event in events_cfg.get("events", [])
        if event.get("active", True) and event.get("formal", False)
    ],
    key=lambda event: int(event.get("formal_order", 999)),
)
assert len(formal_events) == 6, f"Se esperaban 6 eventos formales y se encontraron {len(formal_events)}"

event_order = {
    str(event["event_id"]): int(event["formal_order"])
    for event in formal_events
}

source_posts_path = INTERIM_DIR / "source_posts_candidates_formal.csv"
if not source_posts_path.exists():
    raise FileNotFoundError(f"No existe {source_posts_path}. Ejecuta primero el notebook 01.")

source_candidates_df = pd.read_csv(
    source_posts_path,
    dtype={
        "tweet_id": "string",
        "id": "string",
        "source_post_id": "string",
        "event_id": "string",
        "media_id": "string",
    },
)
if source_candidates_df.empty:
    raise ValueError("source_posts_candidates_formal.csv esta vacio")

before_filters = len(source_candidates_df)
if EVENT_ID_FILTER:
    source_candidates_df = source_candidates_df[
        source_candidates_df["event_id"].astype(str).isin(EVENT_ID_FILTER)
    ].copy()
if MEDIA_ID_FILTER:
    source_candidates_df = source_candidates_df[
        source_candidates_df["media_id"].astype(str).isin(MEDIA_ID_FILTER)
    ].copy()
if source_candidates_df.empty:
    raise ValueError("Los filtros dejaron la muestra formal sin posts")

assert source_candidates_df["source_type"].eq("media_source_post").all()
assert source_candidates_df["source_universe"].eq("costa_rican_media").all()
assert source_candidates_df["media_handle"].notna().all()

print("Filas evento-medio cargadas:", before_filters)
print("Filas despues de filtros:", len(source_candidates_df))
print("Posts madre unicos antes de preparar:", source_candidates_df["source_post_id"].nunique())
display(
    source_candidates_df.groupby(["event_id", "media_id"], dropna=False)
    .size().reset_index(name="n_posts")
    .sort_values(["event_id", "n_posts"], ascending=[True, False])
)


## 4. Manifiestos de recoleccion

Cada post se consulta una sola vez aunque aparezca en dos ventanas formales. Se conserva `formal_event_memberships` para no perder esa relacion.

La query de replies es `conversation_id:{source_post_id} -is:retweet`, sin `lang:es`. El idioma se filtrara posteriormente en pandas.


In [ ]:
urls = xapi.get_x_api_urls()

source_posts_unique_df = col.prepare_source_posts_for_interaction_collection(
    source_posts_df=source_candidates_df,
    event_order=event_order,
)

reply_manifest_df = col.build_reply_collection_manifest(
    source_posts_df=source_candidates_df,
    search_all_url=urls.search_all_url,
    event_order=event_order,
    reply_window_hours=REPLY_WINDOW_HOURS,
    min_reply_count=MIN_SOURCE_REPLY_COUNT,
    max_posts_per_event_media=MAX_POSTS_PER_EVENT_MEDIA,
    max_replies_per_post=MAX_REPLIES_PER_POST,
    max_pages_per_post=MAX_PAGES_PER_POST,
    results_per_page=MAX_RESULTS_PER_REPLY_PAGE,
    batch_size=COLLECTION_BATCH_SIZE,
)

quote_manifest_df = col.build_quote_collection_manifest(
    source_posts_df=source_candidates_df,
    api_base_url=urls.base_url,
    event_order=event_order,
    min_quote_count=MIN_SOURCE_QUOTE_COUNT,
    max_posts_per_event_media=MAX_POSTS_PER_EVENT_MEDIA,
    max_quotes_per_post=MAX_QUOTES_PER_POST,
    max_pages_per_post=MAX_QUOTE_PAGES,
    results_per_page=MAX_RESULTS_PER_QUOTE_PAGE,
    batch_size=COLLECTION_BATCH_SIZE,
)

collection_plan_df = col.summarize_interaction_collection_plan(
    reply_manifest_df=reply_manifest_df,
    quote_manifest_df=quote_manifest_df,
)

batch_frames = []
for manifest in [reply_manifest_df, quote_manifest_df]:
    selected = manifest[manifest["selected_for_collection"]].copy()
    if selected.empty:
        continue
    planned_col = (
        "planned_max_replies"
        if selected["collection_layer"].iloc[0] == "reply_to_media_post"
        else "planned_max_quotes"
    )
    batch_frames.append(
        selected.groupby(["collection_layer", "collection_batch_id"], dropna=False)
        .agg(
            n_posts=("source_post_id", "nunique"),
            reported_interactions=(
                "reply_count" if planned_col == "planned_max_replies" else "quote_count",
                "sum",
            ),
            planned_interaction_cap=(planned_col, "sum"),
            estimated_api_requests=("estimated_api_requests", "sum"),
        )
        .reset_index()
    )
collection_batches_df = (
    pd.concat(batch_frames, ignore_index=True, sort=False)
    if batch_frames
    else pd.DataFrame()
)

SOURCE_UNIQUE_PATH = INTERIM_DIR / "source_posts_formal_unique.csv"
REPLY_MANIFEST_PATH = INTERIM_DIR / "reply_collection_manifest_formal.csv"
QUOTE_MANIFEST_PATH = INTERIM_DIR / "quote_collection_manifest_formal.csv"
PLAN_PATH = OUTPUT_TABLES_DIR / "interaction_collection_plan_formal.csv"
BATCHES_PATH = OUTPUT_TABLES_DIR / "interaction_collection_batches_formal.csv"

source_posts_unique_df.to_csv(SOURCE_UNIQUE_PATH, index=False)
reply_manifest_df.to_csv(REPLY_MANIFEST_PATH, index=False)
quote_manifest_df.to_csv(QUOTE_MANIFEST_PATH, index=False)
collection_plan_df.to_csv(PLAN_PATH, index=False)
collection_batches_df.to_csv(BATCHES_PATH, index=False)

print("[OK]", SOURCE_UNIQUE_PATH)
print("[OK]", REPLY_MANIFEST_PATH)
print("[OK]", QUOTE_MANIFEST_PATH)
print("[OK]", PLAN_PATH)
print("[OK]", BATCHES_PATH)


## 5. Validaciones y costo estimado


In [ ]:
assert len(source_posts_unique_df) == source_posts_unique_df["source_post_id"].nunique()
assert len(reply_manifest_df) == len(source_posts_unique_df)
assert len(quote_manifest_df) == len(source_posts_unique_df)
assert reply_manifest_df["source_post_id"].is_unique
assert quote_manifest_df["source_post_id"].is_unique
assert reply_manifest_df["source_type"].eq("media_source_post").all()
assert reply_manifest_df["source_universe"].eq("costa_rican_media").all()
assert quote_manifest_df["source_type"].eq("media_source_post").all()
assert quote_manifest_df["source_universe"].eq("costa_rican_media").all()
assert reply_manifest_df["query"].str.match(r"^conversation_id:\d+ -is:retweet$").all()
assert ~reply_manifest_df["query"].str.contains("lang:", case=False, regex=False).any()
assert reply_manifest_df["endpoint_url"].eq("https://api.x.com/2/tweets/search/all").all()
assert quote_manifest_df["endpoint_url"].str.match(
    r"^https://api\.x\.com/2/tweets/\d+/quote_tweets$"
).all()
assert reply_manifest_df.loc[
    reply_manifest_df["selected_for_collection"], "start_time"
].notna().all()
assert reply_manifest_df.loc[
    reply_manifest_df["selected_for_collection"], "end_time"
].notna().all()
assert reply_manifest_df.loc[
    reply_manifest_df["selected_for_collection"], "eligible_for_collection"
].all()

n_overlapping = int((source_posts_unique_df["formal_event_count"] > 1).sum())
reply_selected = reply_manifest_df[reply_manifest_df["selected_for_collection"]]
quote_selected = quote_manifest_df[quote_manifest_df["selected_for_collection"]]

print("Posts unicos preparados:", len(source_posts_unique_df))
print("Posts asociados a mas de un evento:", n_overlapping)
print("Replies elegibles:", int(reply_manifest_df["eligible_for_collection"].sum()))
print("Replies seleccionados por el plan:", len(reply_selected))
print("Replies reportados en posts seleccionados:", int(reply_selected["reply_count"].sum()))
print("Tope planificado de replies:", int(reply_selected["planned_max_replies"].sum()))
print("Solicitudes reply estimadas:", int(reply_selected["estimated_api_requests"].sum()))
print("Batches reply:", reply_selected["collection_batch_id"].nunique())
print("Quotes elegibles:", int(quote_manifest_df["eligible_for_collection"].sum()))
print("Quotes seleccionados por el plan:", len(quote_selected))
print("Solicitudes quote estimadas:", int(quote_selected["estimated_api_requests"].sum()))
print("Batches quote:", quote_selected["collection_batch_id"].nunique())

print("\n### Resumen por capa, evento y medio")
display(collection_plan_df)
print("\n### Resumen por batch")
display(collection_batches_df)
print("\n### Top posts seleccionados para replies")
display(
    reply_selected.sort_values(
        ["reply_count", "engagement_score"], ascending=False
    )[[
        "collection_batch_id", "event_id", "formal_event_memberships",
        "media_id", "media_handle", "source_post_id", "reply_count",
        "quote_count", "planned_max_replies", "estimated_api_requests",
        "source_post_text"
    ]].head(30)
)


## 6. Bloqueo de API

La siguiente celda no construye headers en modo dry-run. Para una ejecucion futura se requieren simultaneamente:

- `EXECUTE_X_API_COLLECTION=true`
- `CONFIRM_FORMAL_COLLECTION=HATECR_EXECUTE_FORMAL_X_API`
- un `COLLECTION_BATCH_ID`, por ejemplo `batch_001`

Esto evita que **Run All** consuma recursos accidentalmente.


In [ ]:
headers = None
if API_EXECUTION_ALLOWED:
    xapi.validate_x_api_configuration()
    headers = xapi.build_x_auth_headers()
    print("[ARMED] Credenciales validadas. La recoleccion se limitara al batch indicado.")
else:
    print("[DRY-RUN] API desactivada. No se leyo ni valido X_BEARER_TOKEN para recolectar.")


## 7. Recoleccion futura por batch


In [ ]:
def rows_for_execution(manifest_df):
    selected = manifest_df[manifest_df["selected_for_collection"]].copy()
    if COLLECTION_BATCH_ID:
        selected = selected[
            selected["collection_batch_id"].astype(str).eq(COLLECTION_BATCH_ID)
        ].copy()
    elif not ALLOW_ALL_BATCHES:
        return selected.iloc[0:0].copy()
    return selected.reset_index(drop=True)


replies_clean_df = pd.DataFrame()
replies_raw_rows = []
replies_stats_df = pd.DataFrame()
quote_posts_clean_df = pd.DataFrame()
quote_posts_raw_rows = []
quote_posts_stats_df = pd.DataFrame()

execution_label = COLLECTION_BATCH_ID or "all_batches"
execution_interim_dir = FORMAL_COLLECTION_DIR / execution_label
execution_raw_dir = FORMAL_RAW_DIR / execution_label
if API_EXECUTION_ALLOWED:
    existing_execution_files = [
        path for base in [execution_interim_dir, execution_raw_dir]
        if base.exists()
        for path in base.rglob("*")
        if path.is_file()
    ]
    if existing_execution_files and not ALLOW_BATCH_OVERWRITE:
        raise RuntimeError(
            f"El batch {execution_label} ya tiene archivos. "
            "Usa otro batch o activa ALLOW_BATCH_OVERWRITE=true de forma explicita."
        )
    execution_interim_dir.mkdir(parents=True, exist_ok=True)
    execution_raw_dir.mkdir(parents=True, exist_ok=True)
    print("Directorio interim del batch:", execution_interim_dir)
    print("Directorio raw del batch:", execution_raw_dir)

if API_EXECUTION_ALLOWED and ENABLE_REPLIES_TO_MEDIA_POSTS:
    reply_posts_to_collect_df = rows_for_execution(reply_manifest_df)
    if reply_posts_to_collect_df.empty:
        raise ValueError("El batch seleccionado no contiene posts para replies")
    hash_salt = os.getenv("HASH_SALT", "").strip()
    if not hash_salt:
        raise ValueError("Falta HASH_SALT para anonimizar author_id")

    print("Posts del batch para replies:", len(reply_posts_to_collect_df))
    replies_clean_df, replies_raw_rows, replies_stats_df = col.collect_replies_for_source_posts(
        source_posts_df=reply_posts_to_collect_df,
        headers=headers,
        urls=urls,
        hash_salt=hash_salt,
        use_full_archive=True,
        reply_window_hours=REPLY_WINDOW_HOURS,
        max_replies_per_post=MAX_REPLIES_PER_POST,
        max_pages_per_post=MAX_PAGES_PER_POST,
        max_results_per_page=MAX_RESULTS_PER_REPLY_PAGE,
        timeout_seconds=X_TIMEOUT_SECONDS,
        sleep_seconds=X_SLEEP_SECONDS,
        max_rate_limit_wait_seconds=X_MAX_RATE_WAIT_SECONDS,
        max_429_retries=X_MAX_429_RETRIES,
        recent_days=7,
        allow_recent_fallback=False,
        checkpoint_every=CHECKPOINT_EVERY,
        interim_dir=execution_interim_dir,
        raw_dir=execution_raw_dir,
        stop_on_error=STOP_ON_API_ERROR,
    )
else:
    print("[SKIP] Replies no recolectados; los archivos existentes no se modificaron.")

if API_EXECUTION_ALLOWED and ENABLE_QUOTES_OF_MEDIA_POSTS:
    quote_posts_to_collect_df = rows_for_execution(quote_manifest_df)
    if quote_posts_to_collect_df.empty:
        raise ValueError("El batch seleccionado no contiene posts para quotes")

    print("Posts del batch para quotes:", len(quote_posts_to_collect_df))
    quote_posts_clean_df, quote_posts_raw_rows, quote_posts_stats_df = col.collect_quote_tweets_for_source_posts(
        source_posts_df=quote_posts_to_collect_df,
        headers=headers,
        urls=urls,
        min_quote_count=MIN_SOURCE_QUOTE_COUNT,
        max_quotes_per_post=MAX_QUOTES_PER_POST,
        max_pages_per_post=MAX_QUOTE_PAGES,
        max_results_per_page=MAX_RESULTS_PER_QUOTE_PAGE,
        timeout_seconds=X_TIMEOUT_SECONDS,
        sleep_seconds=X_SLEEP_SECONDS,
        max_rate_limit_wait_seconds=X_MAX_RATE_WAIT_SECONDS,
        max_429_retries=X_MAX_429_RETRIES,
        checkpoint_every=CHECKPOINT_EVERY,
        interim_dir=execution_interim_dir,
        raw_dir=execution_raw_dir,
        stop_on_error=STOP_ON_API_ERROR,
    )
else:
    print("[SKIP] Quotes no recolectados; los archivos existentes no se modificaron.")


## 8. Diagnosticos posteriores a una ejecucion real


In [ ]:
if API_EXECUTION_ALLOWED:
    print("Replies recolectados:", len(replies_clean_df))
    print("Registros raw replies:", len(replies_raw_rows))
    print("Stats replies:", len(replies_stats_df))
    print("Quotes recolectados:", len(quote_posts_clean_df))
    print("Stats quotes:", len(quote_posts_stats_df))

    if not replies_stats_df.empty:
        endpoint_summary_df = (
            replies_stats_df.groupby(
                ["endpoint_used", "status", "status_code"], dropna=False
            ).agg(
                n_posts=("source_post_id", "nunique"),
                replies_kept=("n_rows_kept", "sum"),
                avg_seconds=("seconds", "mean"),
            ).reset_index()
        )
        display(endpoint_summary_df)

        planned_execution_df = rows_for_execution(reply_manifest_df)[[
            "source_post_id", "event_id", "media_id", "media_handle",
            "reply_count", "planned_max_replies", "collection_batch_id"
        ]].copy()
        reply_collection_diagnostics_df = planned_execution_df.merge(
            replies_stats_df,
            on=["source_post_id", "event_id", "media_id", "media_handle"],
            how="left",
        )
        reply_collection_diagnostics_df["reply_count"] = pd.to_numeric(
            reply_collection_diagnostics_df["reply_count"], errors="coerce"
        ).fillna(0)
        reply_collection_diagnostics_df["n_rows_kept"] = pd.to_numeric(
            reply_collection_diagnostics_df["n_rows_kept"], errors="coerce"
        ).fillna(0)
        reply_collection_diagnostics_df["collected_minus_reported"] = (
            reply_collection_diagnostics_df["n_rows_kept"]
            - reply_collection_diagnostics_df["reply_count"]
        )
        reply_collection_diagnostics_df["collection_ratio"] = (
            reply_collection_diagnostics_df["n_rows_kept"]
            / reply_collection_diagnostics_df["reply_count"].replace(0, pd.NA)
        )
        diagnostics_path = execution_interim_dir / "reply_collection_diagnostics.csv"
        reply_collection_diagnostics_df.to_csv(diagnostics_path, index=False)

        created_at = pd.to_datetime(
            replies_clean_df.get("reply_created_at"), errors="coerce", utc=True
        )
        batch_summary_df = pd.DataFrame([
            {"metric": "batch_id", "value": execution_label},
            {"metric": "source_posts_planned", "value": len(planned_execution_df)},
            {"metric": "source_posts_ok", "value": int((replies_stats_df["status"] == "ok").sum())},
            {"metric": "source_posts_with_replies", "value": int((replies_stats_df["n_rows_kept"] > 0).sum())},
            {"metric": "source_posts_zero_replies", "value": int((replies_stats_df["n_rows_kept"] == 0).sum())},
            {"metric": "replies_clean", "value": len(replies_clean_df)},
            {"metric": "unique_reply_ids", "value": replies_clean_df["reply_id"].nunique()},
            {"metric": "unique_author_hashes", "value": replies_clean_df["reply_author_id_hash"].nunique()},
            {"metric": "empty_texts", "value": int(replies_clean_df["reply_text"].fillna("").str.strip().eq("").sum())},
            {"metric": "min_reply_created_at", "value": str(created_at.min())},
            {"metric": "max_reply_created_at", "value": str(created_at.max())},
            {"metric": "avg_seconds_per_post", "value": float(pd.to_numeric(replies_stats_df["seconds"], errors="coerce").mean())},
        ])
        summary_path = OUTPUT_TABLES_DIR / f"replies_{execution_label}_summary.csv"
        batch_summary_df.to_csv(summary_path, index=False)

        by_event_media_df = (
            replies_clean_df.groupby(
                ["event_id", "anchor_media_id", "anchor_media_handle"],
                dropna=False,
            )
            .agg(
                replies=("reply_id", "size"),
                source_posts=("source_post_id", "nunique"),
                unique_authors=("reply_author_id_hash", "nunique"),
            )
            .reset_index()
        )
        by_event_media_path = OUTPUT_TABLES_DIR / f"replies_{execution_label}_by_event_media.csv"
        by_event_media_df.to_csv(by_event_media_path, index=False)

        print("Diagnosticos:", diagnostics_path)
        print("Resumen:", summary_path)
        print("Evento y medio:", by_event_media_path)
else:
    print("[DRY-RUN] No hay diagnosticos de API porque no se ejecuto ninguna llamada.")


## Advertencia metodologica

- `reply_count` y `quote_count` son indicadores publicados por X, no garantias de recuperacion completa.
- Replies eliminados, protegidos o no disponibles pueden producir diferencias entre la metrica y la descarga.
- Los posts presentes en ventanas superpuestas se consultan una sola vez y conservan todas sus membresias de evento.
- El corpus sigue siendo `media_anchored`: toda reply o quote debe estar anclada a un post de un medio configurado.
- Los quote tweets se obtienen mediante su endpoint especifico; este manifiesto no supone que exista una ventana historica equivalente a Full Archive Search.
